# Feature Engineering — Give Me Some Credit

Validación experimental de 9 features candidatas sobre `features_credit_applications`.

**Metodología:** dos baselines (Random Forest y Logistic Regression) → evaluación individual de cada candidata contra ambos baselines por CV estratificada 5-fold → consolidación de resultados → decisión de migración.

**Criterio de aprobación:** `delta_auc > 0.004` **y** `delta_pr_auc > 0.011` (los `std` del CV baseline). Dos modelos en lugar de uno: el Random Forest captura no-linealidad e interacciones implícitas; la Logística es lineal y explícita. Si una feature aporta, debería hacerlo en al menos uno de los dos.

**Tracking:** MLflow (experimento `credit_scoring_features`, backend `mlflow.db`).

In [1]:
from pathlib import Path
import sys
# Obtiene la ruta absoluta de la raíz del proyecto (subiendo un nivel desde 'notebooks/')
ROOT_DIR = Path().resolve().parent

if str(ROOT_DIR) not in sys.path:
  sys.path.append(str(ROOT_DIR))

In [2]:
import duckdb
import pandas as pd
import importlib
import traceback
import os

import numpy as np
import src.features, src.train
importlib.reload(src.features)
importlib.reload(src.train)

from src.features import build_features, get_feature_columns, DB_PATH
from src.train import baseline_logit, baseline_model, log_run

In [3]:
build_features()
con = duckdb.connect(DB_PATH)

df = con.execute("SELECT * FROM features_credit_applications").df()
con.close()

print(f"Shape: {df.shape}")
print(f"Target rate: {df['target'].mean():.4f}")
print(f"Columns: {get_feature_columns()}")

Shape: (149999, 15)
Target rate: 0.0668
Columns: ['revolving_utilization', 'age', 'times_30_59_dpd', 'times_60_89_dpd', 'times_90_dpd', 'debt_ratio', 'monthly_income', 'open_credit_lines', 'real_estate_loans', 'dependents', 'income_missing_flag', 'dependents_missing_flag', 'high_debt_missing_income_flag', 'total_past_due']


## 1. Baselines

Dos modelos sin tuning ni resampling, con las 14 features base:

- **Random Forest** (300 árboles, `min_samples_leaf=10`): captura no-linealidad e interacciones implícitas.
- **Logistic Regression** (`lbfgs`, `max_iter=2000`): modelo lineal explícito, interpretable.

El gap entre ambos informa cuánta señal es lineal vs no-lineal en el dataset.

In [4]:
X_base = df[get_feature_columns()].copy()
y      = df["target"]

base_metrics_rf = log_run(
    run_name       = "baseline-rf",
    model          = baseline_model(),
    X              = X_base,
    y              = y,
    feature_groups = [],
)
print("Baseline Random Forest:")
print(pd.Series(base_metrics_rf).round(4))

Baseline Random Forest:
auc_mean       0.8614
auc_std        0.0040
pr_auc_mean    0.3976
pr_auc_std     0.0112
gini_mean      0.7228
dtype: float64


In [6]:
base_metrics_logit = log_run(
    run_name       = "baseline-logit",
    model          = baseline_logit(),
    X              = X_base,
    y              = y,
    feature_groups = [],
)
print("\nBaseline Logistic Regression:")
print(pd.Series(base_metrics_logit).round(4))


Baseline Logistic Regression:
auc_mean       0.8532
auc_std        0.0044
pr_auc_mean    0.3811
pr_auc_std     0.0073
gini_mean      0.7064
dtype: float64


**Resultado:** Random Forest AUC ≈ 0.861, Logística AUC ≈ 0.853. Gap de **0.8%** con las mismas 14 features: la señal del dataset es mayormente lineal/monotónica. Ambos son baselines sólidos.

El gap pequeño es también una advertencia: features derivadas que apunten a capturar no-linealidad tendrán poco margen para superar al RF. El umbral de aprobación (`delta_auc > 0.004`) es exigente pero justificado: si la feature no suma ni al modelo más débil, no aporta información nueva.

## 2. Features candidatas

| # | Feature | Fórmula | Hipótesis |
|---|---|---|---|
| 1 | `age_30_60` | `(age >= 30) & (age <= 60)` | Tramo etario con mayor mora absoluta (EDA Sección 8) |
| 2 | `has_dependents` | `dependents > 0` (NaN preservado) | Presencia de dependientes como factor |
| 3 | `w_dpd_1_10_100` | `t30 + 10·t60 + 100·t90` | Jerarquía severa de mora |
| 4 | `w_dpd_1_5_10` | `t30 + 5·t60 + 10·t90` | Jerarquía moderada |
| 5 | `dpd_mult` | `(t30+1)(t60+1)(t90+1)` | Interacción 3-vía no lineal |
| 6 | `income_per_person` | `monthly_income / (dependents + 1)` | Capacidad de pago por persona |
| 7 | `income_over_revolving` | `monthly_income / (revolving_utilization + 1)` | Capacidad residual tras uso revolving |
| 8 | `revolving_x_lines` | `revolving_utilization × open_credit_lines` | Exposición total |
| 9 | `revolving_over_lines` | `revolving_utilization / (open_credit_lines + 1)` | Concentración de uso |

## 3. Evaluacion
Cada feature se agrega individualmente al set base y se evalúa en ambos modelos. Criterio: delta_auc > 0.004 AND delta_pr_auc > 0.011.

In [7]:
def build_feature(name: str, df: pd.DataFrame) -> pd.Series:
    """Construye una feature experimental. NaN preservados para imputar en pipeline."""
    if name == "age_30_60":
        return ((df["age"] >= 30) & (df["age"] <= 60)).astype(int)

    if name == "has_dependents":
        result = (df["dependents"] > 0).astype(float)
        result[df["dependents"].isna()] = np.nan
        return result

    if name == "w_dpd_1_10_100":
        return df["times_30_59_dpd"] + 10*df["times_60_89_dpd"] + 100*df["times_90_dpd"]

    if name == "w_dpd_1_5_10":
        return df["times_30_59_dpd"] + 5*df["times_60_89_dpd"] + 10*df["times_90_dpd"]

    if name == "dpd_mult":
        return (df["times_30_59_dpd"] + 1) * (df["times_60_89_dpd"] + 1) * (df["times_90_dpd"] + 1)

    if name == "income_per_person":
        return df["monthly_income"] / (df["dependents"] + 1)

    if name == "income_over_revolving":
        return df["monthly_income"] / (df["revolving_utilization"] + 1)

    if name == "revolving_x_lines":
        return df["revolving_utilization"] * df["open_credit_lines"]

    if name == "revolving_over_lines":
        return df["revolving_utilization"] / (df["open_credit_lines"] + 1)

    raise ValueError(f"Feature desconocida: {name}")


FEATURES_TO_TEST = [
    "age_30_60",
    "has_dependents",
    "w_dpd_1_10_100",
    "w_dpd_1_5_10",
    "dpd_mult",
    "income_per_person",
    "income_over_revolving",
    "revolving_x_lines",
    "revolving_over_lines",
]

THRESHOLD_AUC    = 0.004
THRESHOLD_PR_AUC = 0.011




## 3. Evaluación individual

Cada feature se agrega al set base por separado y se evalúa en ambos modelos con la misma CV (StratifiedKFold 5-fold, `random_state=42`). Los deltas se comparan contra el baseline del mismo modelo.

In [ ]:
results_rf = [{"run": "baseline-rf", "feature": "none", **base_metrics_rf}]

for feat_name in FEATURES_TO_TEST:
    X_feat = X_base.copy()
    X_feat[feat_name] = build_feature(feat_name, df)

    m = log_run(
        run_name       = f"+ {feat_name} [rf]",
        model          = baseline_model(),
        X              = X_feat,
        y              = y,
        feature_groups = [feat_name],
    )
    results_rf.append({"run": f"+ {feat_name} [rf]", "feature": feat_name, **m})

results_rf_df = pd.DataFrame(results_rf)
results_rf_df["delta_auc"]    = results_rf_df["auc_mean"]    - base_metrics_rf["auc_mean"]
results_rf_df["delta_pr_auc"] = results_rf_df["pr_auc_mean"] - base_metrics_rf["pr_auc_mean"]
results_rf_df["aprueba"]      = (results_rf_df["delta_auc"] > THRESHOLD_AUC) & \
                                (results_rf_df["delta_pr_auc"] > THRESHOLD_PR_AUC)

display(
    results_rf_df[["feature","auc_mean","pr_auc_mean","delta_auc","delta_pr_auc","aprueba"]]
    .round(4)
    .sort_values("delta_auc", ascending=False)
)

In [ ]:
results_logit = [{"run": "baseline-logit", "feature": "none", **base_metrics_logit}]

for feat_name in FEATURES_TO_TEST:
    X_feat = X_base.copy()
    X_feat[feat_name] = build_feature(feat_name, df)

    m = log_run(
        run_name       = f"+ {feat_name} [logit]",
        model          = baseline_logit(),
        X              = X_feat,
        y              = y,
        feature_groups = [feat_name],
    )
    results_logit.append({"run": f"+ {feat_name} [logit]", "feature": feat_name, **m})

results_logit_df = pd.DataFrame(results_logit)
results_logit_df["delta_auc"]    = results_logit_df["auc_mean"]    - base_metrics_logit["auc_mean"]
results_logit_df["delta_pr_auc"] = results_logit_df["pr_auc_mean"] - base_metrics_logit["pr_auc_mean"]
results_logit_df["aprueba"]      = (results_logit_df["delta_auc"] > THRESHOLD_AUC) & \
                                   (results_logit_df["delta_pr_auc"] > THRESHOLD_PR_AUC)

display(
    results_logit_df[["feature","auc_mean","pr_auc_mean","delta_auc","delta_pr_auc","aprueba"]]
    .round(4)
    .sort_values("delta_auc", ascending=False)
)

In [ ]:
comparison = (
    results_rf_df[["feature","delta_auc","delta_pr_auc","aprueba"]]
    .merge(
        results_logit_df[["feature","delta_auc","delta_pr_auc","aprueba"]],
        on="feature", suffixes=("_rf", "_logit")
    )
)
comparison["aprueba_alguno"] = comparison["aprueba_rf"] | comparison["aprueba_logit"]

display(
    comparison.round(4).sort_values("delta_auc_logit", ascending=False)
)

In [ ]:
approved = results_df[results_df["aprueba"]]["feature"].tolist()
print(f"Features aprobadas ({len(approved)}): {approved}")

if len(approved) >= 2:
    X_combo = X_base.copy()
    for f in approved:
        X_combo[f] = build_feature(f, df)

    combo_metrics = log_run(
        run_name       = f"+ combined({'|'.join(approved)})",
        model          = baseline_model(),
        X              = X_combo,
        y              = y,
        feature_groups = approved,
    )
    print("\nCombinación:")
    print(pd.Series(combo_metrics).round(4))
else:
    print("Menos de 2 aprobadas. No aplica combinación.")

## 4. Decisiones

Ninguna de las 9 features candidatas superó el umbral (`delta_auc > 0.004` **y** `delta_pr_auc > 0.011`) en ninguno de los dos modelos evaluados.

| Feature | ΔAUC (RF) | ΔAUC (Logit) | ΔPR-AUC (RF) | ΔPR-AUC (Logit) | Decisión |
|---|---|---|---|---|---|
| `dpd_mult` | +0.0002 | **+0.0025** | +0.0002 | **-0.0022** | Descarta |
| `revolving_over_lines` | -0.0003 | +0.0006 | +0.0003 | -0.0004 | Descarta |
| `age_30_60` | -0.0000 | -0.0001 | +0.0010 | -0.0000 | Descarta |
| `w_dpd_1_10_100` | +0.0000 | +0.0000 | -0.0007 | +0.0000 | Descarta |
| `w_dpd_1_5_10` | -0.0000 | +0.0000 | +0.0002 | +0.0000 | Descarta |
| `income_per_person` | -0.0002 | -0.0001 | +0.0005 | -0.0002 | Descarta |
| `income_over_revolving` | -0.0001 | -0.0001 | +0.0000 | -0.0001 | Descarta |
| `revolving_x_lines` | -0.0001 | +0.0000 | +0.0005 | -0.0003 | Descarta |
| `has_dependents` | -0.0004 | -0.0000 | +0.0006 | -0.0000 | Descarta |

**Hallazgos:**

1. **El feature engineering no aportó sobre las 14 features base.** Los modelos capturan las interacciones y no-linealidades de forma implícita (RF), o la señal subyacente es mayormente lineal (logística con gap de solo 0.8% AUC vs RF).

2. **`dpd_mult` en logística subió AUC (+0.0025) pero bajó PR-AUC (-0.0022).** En scoring crediticio con 6.7% de positivos, priorizamos PR-AUC: mide la concentración en el top de riesgo. Un modelo que sube AUC pero baja PR-AUC está perdiendo precisión justo donde importa. Se descarta.

3. **Todas las features que reconstruyen información ya presente** (sumas ponderadas de `times_*`, ratios de `monthly_income`, interacciones de `revolving` con `open_credit_lines`) quedan absorbidas por el modelo base. Es esperable en un dataset de 150k filas con 14 features informativas.

**Decisión:** `features_credit_applications` queda con las 14 features base. Sin migración a `src/features.py`.

## 5. Conclusión

El feature engineering no aporta valor medible sobre las 14 features base en este dataset. La decisión es no migrar ninguna feature derivada y usar `features_credit_applications` tal cual en el pipeline de modelización.

El hallazgo es informativo: el gap RF-Logit de solo 0.8% AUC sugiere que la señal del dataset es mayormente lineal, y el Random Forest extrae el resto vía interacciones implícitas que las features derivadas no logran superar.

**Próximo paso:** `03_modeling_shap.ipynb` — comparación de modelos con SHAP, calibración y métricas de negocio (KS, Bad Rate por decil, Approval Rate).
